# Candidate recovery metric: discharge disposition

Where a patient goes when they leave hospital is the one EHR field that is explicitly a judgement
about their recovery rather than a by-product of one. Discharge to home versus to a skilled
nursing or rehabilitation facility is a clean, clinically meaningful split available on the day
of discharge.

Its weaknesses are coverage and vocabulary: the field is optional in OMOP, it was renamed between
CDM versions, and its values are a long tail of source-specific concepts. This notebook checks all
three before treating it as a candidate.

**Governance.** Every query aggregates inside BigQuery and returns counts,
quantiles or correlations only, so no participant-level rows enter the notebook. Unsuppressed
figures stay git-ignored while `*_suppressed.png` is committable, and the small-cell threshold
must be confirmed against current All of Us policy before anything is published.

# 1-setup

### Datasets, cohort SQL and output directories

`srp_common` resolves the pinned CDR and cohort index, creates this notebook's git-ignored
figure and cache directories, and builds the cohort plus one index date per patient. The
metric-specific CTE is defined here too, so the whole SQL definition this notebook runs on sits
in one place.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas

_root = Path.cwd().resolve()
while not (_root / "srp_common.py").exists():
    if _root.parent == _root:
        raise RuntimeError(
            "Repo root not found: no srp_common.py at or above "
            f"{Path.cwd().resolve()}. Run this notebook from inside the repository."
        )
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import srp_common
from srp_common import CATEGORICAL, INK, INK_MUTED, SERIES, SUPPRESS_THRESHOLD

cfg = srp_common.init("discharge_disposition")
CDR = cfg.CDR
FIGURES, CACHE_DIR = cfg.FIGURES, cfg.CACHE_DIR
cached_gbq = cfg.cached_gbq

# Which of a patient's surgery-days becomes the index date is still open --
# index_surgery/surgery_day_distribution.ipynb is what settles it. "first" is used
# until then because it is the only rule no post-operative event can influence, and
# every result below is stamped with the rule so a later change is visible.
INDEX_RULE = "first"
WINDOWS = srp_common.DEFAULT_WINDOWS

# OMOP CDM 5.3 spells this discharge_to_concept_id; 5.4 renames it to
# discharged_to_concept_id. cdr_schema.py records that the pinned CDR uses the 5.3
# spelling -- if a later release renames it, change this one constant, not the SQL below.
DISCHARGE_COLUMN = "discharge_to_concept_id"

METRIC_CTE = srp_common.index_visit_cte(cfg) + f""",
    disposition AS (
        SELECT
            iv.person_id,
            vo.{DISCHARGE_COLUMN}            AS discharge_concept_id,
            COALESCE(c.concept_name, 'unmapped or null') AS discharge_concept_name
        FROM index_visit AS iv
        JOIN `{CDR}`.visit_occurrence AS vo
          ON vo.visit_occurrence_id = iv.visit_occurrence_id
        LEFT JOIN `{CDR}`.concept AS c
          ON c.concept_id = vo.{DISCHARGE_COLUMN}
        WHERE iv.rn = 1
    )"""

CTES = cfg.cohort_ctes(INDEX_RULE, extra=METRIC_CTE)

print(f"index rule : {INDEX_RULE}")
print("windows    : " + ", ".join(f"{k} [{lo}, {hi}]" for k, (lo, hi) in WINDOWS.items()))

# Fail here, not several minutes into a BigQuery run, if the CDR does not have
# what this notebook reads. cohort_ctes checks its own tables separately.
cfg.require_columns({
    "concept": ["concept_id", "concept_name"],
    "visit_occurrence": ["person_id", "visit_occurrence_id", "visit_concept_id", "visit_start_date", "visit_end_date", "discharge_to_concept_id"],
})


# 2-coverage

### How many patients have a non-null discharge destination?

An optional field is worthless if most rows are null, and All of Us EHR completeness varies sharply
by contributing site. This count is the metric's real coverage, not the count of index stays.

In [ ]:
coverage_sql = CTES + """
    SELECT
        (SELECT COUNT(*) FROM cohort)                   AS cohort_patients,
        (SELECT COUNT(*) FROM index_visit WHERE rn = 1) AS patients_with_index_visit,
        (SELECT COUNTIF(discharge_concept_id IS NOT NULL AND discharge_concept_id != 0)
         FROM disposition)                              AS patients_with_disposition
"""

cov = cached_gbq(coverage_sql, "discharge_disposition_coverage").iloc[0]
cohort_patients = int(cov["cohort_patients"])
patients_with_disposition = int(cov["patients_with_disposition"])

print(f"cohort patients                   : {cohort_patients:>9,}")
print(f"  ... with an inpatient index stay: {int(cov['patients_with_index_visit']):>9,}")
print(f"  ... with a discharge destination: {patients_with_disposition:>9,}"
      f"   ({patients_with_disposition / cohort_patients:6.1%} of cohort)")

# 3-distribution

### Which destinations appear, and how concentrated are they?

The value set is a long tail of source-specific concepts, so the audit question is whether a few
destinations cover most patients. Sparse destinations are aggregated in the suppressed figure
rather than shown.

In [ ]:
dist_sql = CTES + """
    SELECT
        discharge_concept_name,
        COUNT(*) AS n_patients
    FROM disposition
    WHERE discharge_concept_id IS NOT NULL
      AND discharge_concept_id != 0
    GROUP BY discharge_concept_name
    ORDER BY n_patients DESC
"""

disposition_dist = cached_gbq(dist_sql, "discharge_disposition_distribution")
disposition_dist["share"] = (
    disposition_dist["n_patients"] / disposition_dist["n_patients"].sum()
)

print(f"distinct destinations      : {len(disposition_dist):,}")
print(f"smallest cell              : {srp_common.min_cell(disposition_dist, 'n_patients'):,}")
print(f"top 3 cover                : {disposition_dist['share'].nlargest(3).sum():.1%} of patients")

srp_common.plot_distribution(
    cfg, disposition_dist.nlargest(15, "n_patients"), "discharge_concept_name",
    title="Patients by discharge destination from the index stay",
    xlabel="discharge destination", suppress=True, save_as="discharge_disposition")

### Does a home-versus-facility split give two usable groups?

Collapsing the tail into a binary is what makes this metric modellable, and the split is only
useful if both sides clear the suppression threshold comfortably. The keyword rule below is
deliberately crude and should be replaced with an explicit concept list once the value set above
is known.

In [ ]:
FACILITY_KEYWORDS = ["skilled nursing", "nursing", "rehab", "long term", "long-term",
                     "hospice", "inpatient facility", "intermediate care"]

def is_facility(name):
    """Provisional keyword rule -- replace with an explicit concept-id list."""
    lowered = str(name).lower()
    return any(keyword in lowered for keyword in FACILITY_KEYWORDS)

disposition_dist["group"] = [
    "facility" if is_facility(n) else "home or other"
    for n in disposition_dist["discharge_concept_name"]
]
binary = (disposition_dist.groupby("group", as_index=False)["n_patients"].sum()
          .sort_values("n_patients", ascending=False))
binary["share"] = binary["n_patients"] / binary["n_patients"].sum()

print(binary.to_string(index=False))
smallest_group = int(binary["n_patients"].min())
facility_share = float(binary.loc[binary["group"] == "facility", "share"].sum())

if smallest_group < SUPPRESS_THRESHOLD:
    print(f"\nSmaller group holds {smallest_group:,} patients, below the "
          f"{SUPPRESS_THRESHOLD}-patient threshold -- do not report this split.")
else:
    print(f"\nBoth groups clear the {SUPPRESS_THRESHOLD}-patient threshold; "
          f"facility discharge rate = {facility_share:.1%}")

# 4-verdict

### What is the verdict for this metric?

The audit criteria are recorded as one row in the shared task cache, which is what
`metric_comparison.ipynb` reads to build the cross-metric scorecard. Nothing here is a
participant-level value: the row is counts, fractions and hand-entered judgements.

In [ ]:
srp_common.write_scorecard(
    cfg,
    family="ehr",
    source_table=f"visit_occurrence.{DISCHARGE_COLUMN}",
    index_rule=INDEX_RULE,
    outcome_type="binary",
    cohort_patients=cohort_patients,
    patients_with_metric=patients_with_disposition,
    patients_analysable=patients_with_disposition,
    coverage_share=patients_with_disposition / cohort_patients,
    horizon_days=0,
    event_rate=facility_share,
    informative_missingness="unclear",
    leakage_risk="low",
    notes=("Null-heavy optional field whose completeness varies by contributing site, so its "
           "missingness may track the site rather than the patient. The home/facility split is "
           "currently a keyword rule and needs an explicit concept list before use."),
)